# Synthetic Table-Reasoning Curriculum for WikiTableQuestions

This notebook evaluates untouched Qwen3-1.7B + zero-initialized LoRA on WTQ validation, then follows one continuous LoRA trajectory through synthetic Levels 1–5. Every stage is evaluated with the same official WTQ denotation metric and saved directly to Google Drive.

## 1. Runtime information

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime: {torch.version.cuda}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Clone or update the repository

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
BRANCH = "main"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git commit: {commit}")
%cd /content/table-cnn-mrc

## 3. Install repository dependencies

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)

## 4. Mount Drive and verify all synthetic datasets

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

DRIVE_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr")
CURRICULUM_OUTPUT = DRIVE_ROOT / "outputs/synthetic_curriculum"
OFFICIAL_CACHE = DRIVE_ROOT / "outputs/diagnostics/wtq_official_1.0.2"
CURRICULUM_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Persistent curriculum output: {CURRICULUM_OUTPUT}")

In [ ]:
import csv
from collections import Counter

for level in range(1, 6):
    path = REPO_DIR / f"dataset_level_{level}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Missing required repository dataset: {path}")
    with path.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if any(not row.get("prompt", "").strip() or not row.get("answer", "").strip() for row in rows):
        raise ValueError(f"Level {level} contains a null prompt or answer")
    tasks = Counter(row.get("task_type") or row.get("operation") or "unknown" for row in rows)
    print(f"\nLevel {level}: {len(rows)} rows | tasks={dict(tasks)}")
    print(f"First prompt:\n{rows[0]['prompt']}")
    print(f"First answer: {rows[0]['answer']}")

## 5. Experiment configuration

The pure curriculum is the primary experiment. Keep `RUN_MIXED_PHASE=False` initially. Keep `RUN_FINAL_TEST=False` until you have reviewed and selected the validation configuration.

In [ ]:
SEED = 42
BASE_MODEL = "Qwen/Qwen3-1.7B"
NUM_EPOCHS_PER_LEVEL = 3
LEARNING_RATE = 5e-5
MAX_SEQUENCE_LENGTH = 2048
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
CHECKPOINT_EVERY_STEPS = 25
RUN_MIXED_PHASE = False
A_TO_B_RATIO = (25, 75)
MIXED_EPOCHS = 1
RUN_FINAL_TEST = False

print({
    "seed": SEED, "base_model": BASE_MODEL,
    "epochs_per_level": NUM_EPOCHS_PER_LEVEL,
    "learning_rate": LEARNING_RATE,
    "mixed_phase": RUN_MIXED_PHASE, "final_test": RUN_FINAL_TEST,
})

## 6. Load WTQ and print split sizes

In [ ]:
from datasets import load_dataset
wtq = load_dataset("stanfordnlp/wikitablequestions", revision="refs/pr/4")
print(f"WTQ train: {len(wtq['train'])}")
print(f"WTQ validation: {len(wtq['validation'])}")
print(f"WTQ test: {len(wtq['test'])}")
del wtq

## 7. Pipeline smoke test

This checks all five CSVs, formatting, tokenization, one LoRA training step, one WTQ validation example, checkpoint save, and checkpoint reload.

In [ ]:
smoke_command = [
    sys.executable, "-u", str(REPO_DIR / "scripts/smoke_curriculum.py"),
    "--data-root", str(REPO_DIR),
    "--official-cache-dir", str(OFFICIAL_CACHE),
]
smoke_text = " ".join(shlex.quote(str(part)) for part in smoke_command)
smoke_status = Path("/tmp/table_curriculum_smoke_exit_code.txt")
smoke_status.unlink(missing_ok=True)
get_ipython().system(
    f"cd {shlex.quote(str(REPO_DIR))} && PYTHONUNBUFFERED=1 "
    f"TABLE_MRC_PLAIN_PROGRESS=1 TQDM_DISABLE=1 {smoke_text}; "
    f"printf '%s' $? > {shlex.quote(str(smoke_status))}"
)
if not smoke_status.is_file():
    raise RuntimeError("Curriculum smoke-test exit status was not recorded")
smoke_return_code = int(smoke_status.read_text(encoding="utf-8").strip())
if smoke_return_code != 0:
    raise subprocess.CalledProcessError(smoke_return_code, smoke_command)

## 8. Base → L1 → L2 → L3 → L4 → L5

This is one continuous LoRA trajectory. The runner saves mid-stage state every 25 optimizer steps, evaluates WTQ validation after every level, and resumes safely when this cell is rerun.

In [ ]:
command = [
    sys.executable, "-u", str(REPO_DIR / "scripts/run_curriculum.py"),
    "--data-root", str(REPO_DIR),
    "--output-dir", str(CURRICULUM_OUTPUT),
    "--official-cache-dir", str(OFFICIAL_CACHE),
    "--base-model", BASE_MODEL,
    "--epochs-per-level", str(NUM_EPOCHS_PER_LEVEL),
    "--learning-rate", str(LEARNING_RATE),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
    "--checkpoint-every-steps", str(CHECKPOINT_EVERY_STEPS),
    "--seed", str(SEED),
    "--synthetic-ratio", str(A_TO_B_RATIO[0]),
    "--wtq-ratio", str(A_TO_B_RATIO[1]),
    "--mixed-epochs", str(MIXED_EPOCHS),
]
if RUN_MIXED_PHASE:
    command.append("--run-mixed-phase")
if RUN_FINAL_TEST:
    command.append("--run-final-test")
command_text = " ".join(shlex.quote(str(part)) for part in command)
status_path = Path("/tmp/table_curriculum_exit_code.txt")
status_path.unlink(missing_ok=True)
shell_command = (
    f"cd {shlex.quote(str(REPO_DIR))} && "
    f"PYTHONUNBUFFERED=1 TABLE_MRC_PLAIN_PROGRESS=1 TQDM_DISABLE=1 "
    f"{command_text}; printf '%s' $? > {shlex.quote(str(status_path))}"
)
get_ipython().system(shell_command)
if not status_path.is_file():
    raise RuntimeError("Curriculum exit status was not recorded")
return_code = int(status_path.read_text(encoding="utf-8").strip())
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

## 9. Results table and validation-transfer curve

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

results_path = CURRICULUM_OUTPUT / "results/curriculum_results.csv"
summary_path = CURRICULUM_OUTPUT / "results/curriculum_summary.json"
results = pd.read_csv(results_path)
display(results)
with summary_path.open(encoding="utf-8") as handle:
    summary = json.load(handle)
print(json.dumps(summary, indent=2))
mixed_path = CURRICULUM_OUTPUT / "results/mixed_result.json"
if mixed_path.is_file():
    with mixed_path.open(encoding="utf-8") as handle:
        print("Mixed-phase result:")
        print(json.dumps(json.load(handle), indent=2))
plt.figure(figsize=(8, 4))
plt.plot(results["stage"], results["wtq_validation_score"], marker="o")
plt.xlabel("Curriculum stage")
plt.ylabel("WTQ validation denotation accuracy")
plt.title("Synthetic table-reasoning transfer curve")
plt.grid(alpha=0.3)
plt.show()

## 10. Optional mixed phase

To run synthetic + WTQ training, set `RUN_MIXED_PHASE=True` in the configuration cell and rerun the main experiment cell. Completed pure stages are not repeated. The ratio is controlled by `A_TO_B_RATIO`.

## 11. One-time final WTQ test

After reviewing validation results, set `RUN_FINAL_TEST=True` and rerun the main experiment cell. It selects the best validation checkpoint and evaluates WTQ test once. If `final_test.json` already exists, the recorded result is retained instead of testing again.

In [ ]:
final_test_path = CURRICULUM_OUTPUT / "results/final_test.json"
if final_test_path.is_file():
    with final_test_path.open(encoding="utf-8") as handle:
        print(json.dumps(json.load(handle), indent=2))
else:
    print("Final test has not been run. Select by validation first.")